In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read bronze forex data
print("📖 Reading bronze forex data...")
bronze_forex = spark.table("financial_market.bronze.forex")

print(f"Initial record count: {bronze_forex.count():,}")

# 1. Convert string dates to proper date type
print("\n🔄 Converting date columns to proper types...")
cleaned_forex = bronze_forex \
    .withColumn("rate_date", F.to_date(F.col("rate_date")))

# 2. Remove records with null critical values
print("\n🧹 Removing records with null critical values...")
initial_count = cleaned_forex.count()
cleaned_forex = cleaned_forex.filter(
    F.col("base_currency").isNotNull() &
    F.col("target_currency").isNotNull() &
    F.col("exchange_rate").isNotNull() &
    F.col("rate_date").isNotNull() &
    F.col("fetch_timestamp").isNotNull()
)
null_removed = initial_count - cleaned_forex.count()
print(f"  Removed {null_removed:,} records with null values")

# 3. Remove records with invalid exchange rates
print("\n🔍 Removing records with invalid exchange rates...")
initial_count = cleaned_forex.count()
cleaned_forex = cleaned_forex.filter(
    (F.col("exchange_rate") > 0) &
    (F.col("exchange_rate") < 1000000)  # Reasonable upper bound
)
invalid_removed = initial_count - cleaned_forex.count()
print(f"  Removed {invalid_removed:,} records with invalid rates")

# 4. Remove duplicate records (keep most recent fetch)
print("\n🔄 Removing duplicates (keeping most recent fetch)...")
initial_count = cleaned_forex.count()
window_spec = Window.partitionBy("base_currency", "target_currency", "rate_date").orderBy(F.col("fetch_timestamp").desc())
cleaned_forex = cleaned_forex \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")
duplicates_removed = initial_count - cleaned_forex.count()
print(f"  Removed {duplicates_removed:,} duplicate records")

# 5. Add calculated columns for analysis
print("\n➕ Adding calculated columns...")
cleaned_forex = cleaned_forex \
    .withColumn("inverse_rate", F.round(1 / F.col("exchange_rate"), 6)) \
    .withColumn("inverse_pair", F.concat(F.col("target_currency"), F.lit("/"), F.col("base_currency")))

# 6. Add data quality flag
print("\n✅ Adding data quality indicators...")
cleaned_forex = cleaned_forex \
    .withColumn("data_quality", F.lit("clean")) \
    .withColumn("processed_at", F.current_timestamp())

# Display summary statistics
print("\n📊 Summary Statistics:")
print(f"Final record count: {cleaned_forex.count():,}")
print(f"Date range: {cleaned_forex.agg(F.min('rate_date')).collect()[0][0]} to {cleaned_forex.agg(F.max('rate_date')).collect()[0][0]}")
print(f"Unique currency pairs: {cleaned_forex.select('pair').distinct().count()}")
print(f"Base currencies: {cleaned_forex.select('base_currency').distinct().count()}")
print(f"Target currencies: {cleaned_forex.select('target_currency').distinct().count()}")

# Show sample
print("\n📋 Sample of cleaned forex data:")
display(cleaned_forex.orderBy(F.col("rate_date").desc()).limit(10))

In [0]:
# Silver table mein save karo
print("💾 Saving to Silver layer...")

cleaned_forex.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("base_currency") \
    .saveAsTable("financial_market.silver.forex")

print(f"✅ Saved to: financial_market.silver.forex")

# verify
df_verify = spark.sql("""
    SELECT
        base_currency,
        target_currency,
        exchange_rate,
        inverse_rate,
        rate_date,
        data_quality
    FROM financial_market.silver.forex
    ORDER BY target_currency
""")

print(f"✅ Total rows: {df_verify.count()}")
display(df_verify)